# Event Stream Generation

In [18]:
import cv2
import numpy as np

import os

In [19]:
from images import process_frame, build_event_image

In [ ]:
IMAGES_PATH = "../data/thun_00_a/thun_00_a_images_rectified_left"

In [ ]:
VIDEO_PATH = os.path.join(IMAGES_PATH, "thun_00_a.mp4")
cap = cv2.VideoCapture(VIDEO_PATH)

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(width, height)
fps = cap.get(cv2.CAP_PROP_FPS)

In [ ]:
VIDEO_OUTPUT_PATH = "../outputs/thun_00_a_generated.mp4"

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(VIDEO_OUTPUT_PATH, fourcc, fps, (width, height))

In [ ]:
# Intensity change threshold for event generation
THRESHOLD = 0.2 
# Minimum number of motion pixels to consider for rendering an event frame
MIN_MOTION_PIXELS_TO_RENDER = 500

ret, frame = cap.read()

last_event_image = process_frame(frame)
generated_events = []

In [ ]:
cnt = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    # Grayscale conversion and logarithmic transformation
    current_image = process_frame(frame)
    # Noise reduction 
    current_image_blurred = cv2.GaussianBlur(current_image, (5, 5), 0)
    diff = current_image_blurred.astype(np.int16) - last_event_image.astype(np.int16)
    motion_mask = np.abs(diff) > THRESHOLD
    motion_pixels = np.argwhere(motion_mask)
    if len(motion_pixels) > MIN_MOTION_PIXELS_TO_RENDER:
        timestamp = cap.get(cv2.CAP_PROP_POS_MSEC)
        rows = motion_pixels[:, 0] 
        cols = motion_pixels[:, 1] 
        polarities = np.where(diff[rows, cols] > 0, 1, -1)
        events = np.column_stack((
            cols, 
            rows, 
            np.full(len(rows), timestamp),
            polarities
        ))
        generated_events.extend(events) 
        last_event_image[rows, cols] = current_image[rows, cols]
        cnt += 1
        event_img = build_event_image(events, height, width)
        out.write(event_img) 

print(cnt)
cap.release()
out.release()

In [ ]:
print(len(generated_events))

generated_events = np.array(generated_events)
print(len(generated_events))

np.save("../outputs/thun_00_a_generated_events.npy", generated_events)